# Phase 1 Colab runner: where does forgetting live?

All logic lives in `src/`. Runtime: GPU (T4).

Two separate experiments, as in the plan: **A. Flowers102** as the target, **B. EuroSAT** as the target. Each fine-tunes its own models (3 seeds), then reverts / retains each component.

Every step is separate and ends with a push, so a disconnect loses at most one step. In a new session, rerun **Setup** first, then continue where you stopped.

## Setup (every session)

In [3]:
from google.colab import drive, userdata
drive.mount('/content/drive')

REPO = 'sohilkhan-2611/-clip-zero-shot-forgetting'
BRANCH = 'phase-1'
CKPT_DIR = '/content/drive/MyDrive/clip-forgetting/checkpoints'
DATA_DIR = '/content/data'   # local disk, re-downloaded each session
token = userdata.get('GITHB-TOKEN')

Mounted at /content/drive


In [4]:
import os
if not os.path.exists('/content/repo'):
    !git clone -q -b {BRANCH} https://{token}@github.com/{REPO}.git /content/repo
%cd /content/repo
!git pull -q
!pip install -q -e .[dev]
!nvidia-smi --query-gpu=name,memory.total --format=csv
!ls -la {CKPT_DIR}

/content/repo
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for clip-forgetting-deltas (pyproject.toml) ... done
name, memory.total [MiB]
Tesla T4, 15360 MiB
total 332302
drwx------ 2 root root      4096 Nov 14 01:16 .
drwx------ 3 root root      4096 Nov 14 01:16 ..
-rw------- 1 root root 170134399 Nov 14 01:16 finetune_flowers_vitb16_seed0.pt
-rw------- 1 root root 170134399 Nov 14 03:08 finetune_flowers_vitb16_seed1.pt


## A1. Flowers102: fine-tune seeds 1 and 2
Seed 0 already exists from Phase 0 (check it is listed above). Each run takes roughly 35 minutes (Phase 0 timing) and writes its checkpoint to Drive.

In [4]:
!python -m src.finetune --config configs/finetune_flowers_vitb16.yaml --set seed=1 data.root={DATA_DIR} output.checkpoint_dir={CKPT_DIR} output.results_dir=results/phase1


open_clip_model.safetensors: downloading bytes:  72% 428M/599M [00:02<00:00, 174MB/s, 37.0MB/s  ]
open_clip_model.safetensors: reconstructing file:  78% 470M/599M [00:04<00:01, 103MB/s]
open_clip_model.safetensors: downloading bytes: 100% 428M/428M [00:05<00:00, 84.8MB/s, 37.7MB/s  ]
open_clip_model.safetensors: reconstructing file: 100% 599M/599M [00:05<00:00, 119MB/s, 51.9MB/s  ]
trainable tensors: 72, parameters: 42.53M
100% 345M/345M [00:25<00:00, 13.7MB/s]
100% 502/502 [00:00<00:00, 2.20MB/s]
100% 15.0k/15.0k [00:00<00:00, 65.5MB/s]
/usr/local/lib/python3.13/dist-packages/clip_benchmark/datasets/builder.py:425: UserWarning: split argument ignored for `eurosat`, there are no pre-defined train/test splits for this dataset
  warnings.warn(f"split argument ignored for `{dataset_name}`, there are no pre-defined train/test splits for this dataset")
100% 94.3M/94.3M [00:01<00:00, 70.1MB/s]
100% 170M/170M [00:08<00:00, 20.5MB/s]
100% 169M/169M [00:18<00:00, 9.33MB/s]
100% 625M/625M [00:3

In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
import os

CKPT = '/content/drive/MyDrive/clip-forgetting/checkpoints/finetune_flowers_vitb16_seed1.pt'

print("Checkpoint exists:", os.path.exists(CKPT))

Checkpoint exists: True


In [5]:
!git config user.name 'colab-runner'
!git config user.email 'colab-runner@users.noreply.github.com'
!git add results/phase1/finetune_*.json
!git commit -q -m 'Phase 1: Flowers102 fine-tune, seed 1' || echo 'nothing to commit'
!git pull -q --rebase origin {BRANCH}
!git push -q origin {BRANCH}
!git status -sb | head -1
!git log --oneline -1

fatal: pathspec 'results/phase1/finetune_*.json' did not match any files
On branch phase-1
Your branch is up to date with 'origin/phase-1'.

nothing to commit, working tree clean
nothing to commit
## phase-1...origin/phase-1
d1e248f (HEAD -> phase-1, origin/phase-1) Phase 1: add EuroSAT as second target, as in the plan


In [ ]:
!python -m src.finetune --config configs/finetune_flowers_vitb16.yaml --set seed=2 data.root={DATA_DIR} output.checkpoint_dir={CKPT_DIR} output.results_dir=results/phase1

In [ ]:
!git config user.name 'colab-runner'
!git config user.email 'colab-runner@users.noreply.github.com'
!git add results/phase1/finetune_*.json
!git commit -q -m 'Phase 1: Flowers102 fine-tune, seed 2' || echo 'nothing to commit'
!git pull -q --rebase origin {BRANCH}
!git push -q origin {BRANCH}
!git status -sb | head -1
!git log --oneline -1

## A2. Flowers102: revert / retain per component
One checkpoint per cell: 10 conditions (pre, ft, 4 revert, 4 retain), each evaluated on Flowers (2,000 images) and the five held-out sets. Results are saved after every condition; if a cell stops, just run it again and it continues where it stopped.

For seed 0, the `ft` numbers should equal the Phase 0 fine-tune result exactly. That is a free consistency check.

In [ ]:
!python -m src.components --config configs/phase1_components_flowers.yaml --set data.root={DATA_DIR} checkpoints=[{CKPT_DIR}/finetune_flowers_vitb16_seed0.pt]

In [ ]:
!git config user.name 'colab-runner'
!git config user.email 'colab-runner@users.noreply.github.com'
!git add results/phase1/components_*.json
!git commit -q -m 'Phase 1: components, seed 0' || echo 'nothing to commit'
!git pull -q --rebase origin {BRANCH}
!git push -q origin {BRANCH}
!git status -sb | head -1
!git log --oneline -1

In [ ]:
!python -m src.components --config configs/phase1_components_flowers.yaml --set data.root={DATA_DIR} checkpoints=[{CKPT_DIR}/finetune_flowers_vitb16_seed1.pt]

In [ ]:
!git config user.name 'colab-runner'
!git config user.email 'colab-runner@users.noreply.github.com'
!git add results/phase1/components_*.json
!git commit -q -m 'Phase 1: components, seed 1' || echo 'nothing to commit'
!git pull -q --rebase origin {BRANCH}
!git push -q origin {BRANCH}
!git status -sb | head -1
!git log --oneline -1

In [ ]:
!python -m src.components --config configs/phase1_components_flowers.yaml --set data.root={DATA_DIR} checkpoints=[{CKPT_DIR}/finetune_flowers_vitb16_seed2.pt]

In [ ]:
!git config user.name 'colab-runner'
!git config user.email 'colab-runner@users.noreply.github.com'
!git add results/phase1/components_*.json
!git commit -q -m 'Phase 1: components, seed 2' || echo 'nothing to commit'
!git pull -q --rebase origin {BRANCH}
!git push -q origin {BRANCH}
!git status -sb | head -1
!git log --oneline -1

## A3. Flowers102: table over seeds

In [ ]:
!python -m src.aggregate_phase1 --inputs 'results/phase1/components_flowers_*.json' --out results/phase1/summary_components_flowers.json

In [ ]:
!git config user.name 'colab-runner'
!git config user.email 'colab-runner@users.noreply.github.com'
!git add results/phase1/summary_*.json
!git commit -q -m 'Phase 1: summary table over seeds' || echo 'nothing to commit'
!git pull -q --rebase origin {BRANCH}
!git push -q origin {BRANCH}
!git status -sb | head -1
!git log --oneline -1

## B1. EuroSAT: fine-tune seeds 0, 1 and 2
EuroSAT has about 10 times more training images than Flowers (21,600 vs 2,040), so each run takes longer than the Flowers ones. Here EuroSAT is the target and Flowers becomes a held-out set.

In [ ]:
!python -m src.finetune --config configs/finetune_eurosat_vitb16.yaml --set seed=0 data.root={DATA_DIR} output.checkpoint_dir={CKPT_DIR} output.results_dir=results/phase1

In [ ]:
!git config user.name 'colab-runner'
!git config user.email 'colab-runner@users.noreply.github.com'
!git add results/phase1/finetune_*.json
!git commit -q -m 'Phase 1: EuroSAT fine-tune, seed 0' || echo 'nothing to commit'
!git pull -q --rebase origin {BRANCH}
!git push -q origin {BRANCH}
!git status -sb | head -1
!git log --oneline -1

In [ ]:
!python -m src.finetune --config configs/finetune_eurosat_vitb16.yaml --set seed=1 data.root={DATA_DIR} output.checkpoint_dir={CKPT_DIR} output.results_dir=results/phase1

In [ ]:
!git config user.name 'colab-runner'
!git config user.email 'colab-runner@users.noreply.github.com'
!git add results/phase1/finetune_*.json
!git commit -q -m 'Phase 1: EuroSAT fine-tune, seed 1' || echo 'nothing to commit'
!git pull -q --rebase origin {BRANCH}
!git push -q origin {BRANCH}
!git status -sb | head -1
!git log --oneline -1

In [ ]:
!python -m src.finetune --config configs/finetune_eurosat_vitb16.yaml --set seed=2 data.root={DATA_DIR} output.checkpoint_dir={CKPT_DIR} output.results_dir=results/phase1

In [ ]:
!git config user.name 'colab-runner'
!git config user.email 'colab-runner@users.noreply.github.com'
!git add results/phase1/finetune_*.json
!git commit -q -m 'Phase 1: EuroSAT fine-tune, seed 2' || echo 'nothing to commit'
!git pull -q --rebase origin {BRANCH}
!git push -q origin {BRANCH}
!git status -sb | head -1
!git log --oneline -1

## B2. EuroSAT: revert / retain per component

In [ ]:
!python -m src.components --config configs/phase1_components_eurosat.yaml --set data.root={DATA_DIR} checkpoints=[{CKPT_DIR}/finetune_eurosat_vitb16_seed0.pt]

In [ ]:
!git config user.name 'colab-runner'
!git config user.email 'colab-runner@users.noreply.github.com'
!git add results/phase1/components_*.json
!git commit -q -m 'Phase 1: EuroSAT components, seed 0' || echo 'nothing to commit'
!git pull -q --rebase origin {BRANCH}
!git push -q origin {BRANCH}
!git status -sb | head -1
!git log --oneline -1

In [ ]:
!python -m src.components --config configs/phase1_components_eurosat.yaml --set data.root={DATA_DIR} checkpoints=[{CKPT_DIR}/finetune_eurosat_vitb16_seed1.pt]

In [ ]:
!git config user.name 'colab-runner'
!git config user.email 'colab-runner@users.noreply.github.com'
!git add results/phase1/components_*.json
!git commit -q -m 'Phase 1: EuroSAT components, seed 1' || echo 'nothing to commit'
!git pull -q --rebase origin {BRANCH}
!git push -q origin {BRANCH}
!git status -sb | head -1
!git log --oneline -1

In [ ]:
!python -m src.components --config configs/phase1_components_eurosat.yaml --set data.root={DATA_DIR} checkpoints=[{CKPT_DIR}/finetune_eurosat_vitb16_seed2.pt]

In [ ]:
!git config user.name 'colab-runner'
!git config user.email 'colab-runner@users.noreply.github.com'
!git add results/phase1/components_*.json
!git commit -q -m 'Phase 1: EuroSAT components, seed 2' || echo 'nothing to commit'
!git pull -q --rebase origin {BRANCH}
!git push -q origin {BRANCH}
!git status -sb | head -1
!git log --oneline -1

## B3. EuroSAT: table over seeds

In [ ]:
!python -m src.aggregate_phase1 --inputs 'results/phase1/components_eurosat_*.json' --out results/phase1/summary_components_eurosat.json

In [ ]:
!git config user.name 'colab-runner'
!git config user.email 'colab-runner@users.noreply.github.com'
!git add results/phase1/summary_*.json
!git commit -q -m 'Phase 1: EuroSAT summary table over seeds' || echo 'nothing to commit'
!git pull -q --rebase origin {BRANCH}
!git push -q origin {BRANCH}
!git status -sb | head -1
!git log --oneline -1